In [1]:
import pandas as pd
import hashlib
import json
from pathlib import Path
from datetime import datetime

In [2]:
# import shutil
# from pathlib import Path
# from datetime import datetime

# # Paths
# HISTORY_PATH = Path("selection_history.json")
# TRAIN_DIR    = Path("training_sets")
# OUTPUT_DIR   = Path(".")

# # Backup folder with timestamp
# BACKUP_DIR = Path("backup_before_reset") / datetime.now().strftime("%Y%m%d_%H%M%S")
# BACKUP_DIR.mkdir(parents=True, exist_ok=True)

# # Move history file if present
# if HISTORY_PATH.exists():
#     shutil.move(str(HISTORY_PATH), BACKUP_DIR / HISTORY_PATH.name)

# # Move previous training sets
# if TRAIN_DIR.exists():
#     shutil.move(str(TRAIN_DIR), BACKUP_DIR / TRAIN_DIR.name)

# # Move any unique_sample_*.csv files
# moved_any = False
# for p in OUTPUT_DIR.glob("unique_sample_*.csv"):
#     shutil.move(str(p), BACKUP_DIR / p.name)
#     moved_any = True

# print("✅ Reset complete. Archived prior state to:", BACKUP_DIR.resolve())

import shutil
from pathlib import Path
from datetime import datetime

# Paths (fixed leading slashes)
HISTORY_PATH = Path("/home/ubuntu/TW_MultiLabel_SMP/jupyter-notebooks/selection_history.json")
TRAIN_DIR    = Path("/home/ubuntu/TW_MultiLabel_SMP/datasets/training_sets")
OUTPUT_DIR   = Path(".")

# Backup folder with timestamp
BACKUP_DIR = Path("/home/ubuntu/TW_MultiLabel_SMP/datasets/backup_before_reset") / datetime.now().strftime("%Y%m%d_%H%M%S")
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

# ---- Preserve history (copy, don't move) ----
if HISTORY_PATH.exists():
    backup_history = BACKUP_DIR / HISTORY_PATH.name
    try:
        shutil.copy2(HISTORY_PATH, backup_history)
        print(f"📝 Preserved history: copied to {backup_history}")
    except Exception as e:
        print(f"⚠️ Could not copy history file: {e}")
else:
    print("ℹ️ No selection_history.json found to preserve.")

# ---- Move previous training sets to backup ----
if TRAIN_DIR.exists():
    dest = BACKUP_DIR / TRAIN_DIR.name
    try:
        shutil.move(str(TRAIN_DIR), dest)
        print(f"📦 Archived training_sets to: {dest}")
    except Exception as e:
        print(f"⚠️ Could not move training_sets: {e}")
else:
    print("ℹ️ No training_sets directory found to archive.")

# ---- Move any unique_sample_*.csv files to backup ----
moved_any = False
for p in OUTPUT_DIR.glob("unique_sample_*.csv"):
    try:
        shutil.move(str(p), BACKUP_DIR / p.name)
        print(f"📄 Archived {p.name}")
        moved_any = True
    except Exception as e:
        print(f"⚠️ Could not move {p}: {e}")

if not moved_any:
    print("ℹ️ No unique_sample_*.csv files found to archive.")

print("✅ Reset complete. Old artifacts archived. History preserved in place.")



📝 Preserved history: copied to /home/ubuntu/TW_MultiLabel_SMP/datasets/backup_before_reset/20251020_214414/selection_history.json
ℹ️ No training_sets directory found to archive.
📄 Archived unique_sample_20251018_212534.csv
✅ Reset complete. Old artifacts archived. History preserved in place.


In [3]:
def row_hash(row: pd.Series) -> str:
    """Generate a deterministic hash of a row if no explicit ID column exists."""
    obj = row.to_dict()
    normalized = {str(k): ("" if pd.isna(v) else str(v)) for k, v in obj.items()}
    payload = json.dumps(normalized, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

def load_df(path: str, dataset_name: str, id_column: str = None) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)
    df["__dataset"] = dataset_name
    df["__source_file"] = Path(path).name

    if id_column and id_column in df.columns:
        df["unique_key"] = df[id_column].astype(str)
    else:
        df["unique_key"] = df.apply(row_hash, axis=1)

    return df


In [4]:
# Update these paths to your local copies
ABORTION_PATH = "/home/ubuntu/TW_MultiLabel_SMP/datasets/abortion_data-updated - new_abortion_related_subreddits_text_posts .csv"
MISCARRIAGE_PATH = "/home/ubuntu/TW_MultiLabel_SMP/datasets/miscarriage_related_posts.csv"
HARASSMENT_PATH = "/home/ubuntu/TW_MultiLabel_SMP/datasets/sexual-harrassment-data-updated - RelevantByTitle.csv"

# History file (persists across runs)
HISTORY_PATH = Path("/home/ubuntu/TW_MultiLabel_SMP/jupyter-notebooks/selection_history.json")

# Desired counts per dataset
counts = {
    "abortion": 167,
    "miscarriage": 166,
    "harassment": 167
}

# Optional: if your CSVs have a post_id or id column
ID_COLUMN = None   # e.g. "post_id"


In [5]:
# Load CSVs
abortion_df = load_df(ABORTION_PATH, "abortion", ID_COLUMN)
miscarriage_df = load_df(MISCARRIAGE_PATH, "miscarriage", ID_COLUMN)
harassment_df = load_df(HARASSMENT_PATH, "harassment", ID_COLUMN)

# Load or initialize selection history
if HISTORY_PATH.exists():
    with open(HISTORY_PATH, "r", encoding="utf-8") as f:
        history = json.load(f)
else:
    history = {"used_keys": [], "runs": []}

used_keys = set(history.get("used_keys", []))

# Exclude previously used posts
def exclude_used(df):
    return df[~df["unique_key"].isin(used_keys)].copy()

ab_pool = exclude_used(abortion_df)
mi_pool = exclude_used(miscarriage_df)
sh_pool = exclude_used(harassment_df)

In [6]:
shortages = []
if len(ab_pool) < counts["abortion"]:
    shortages.append(f"abortion (need {counts['abortion']}, have {len(ab_pool)})")
if len(mi_pool) < counts["miscarriage"]:
    shortages.append(f"miscarriage (need {counts['miscarriage']}, have {len(mi_pool)})")
if len(sh_pool) < counts["harassment"]:
    shortages.append(f"harassment (need {counts['harassment']}, have {len(sh_pool)})")

if shortages:
    raise RuntimeError("Not enough fresh rows: " + "; ".join(shortages))

sample_ab = ab_pool.sample(n=counts["abortion"], replace=False, random_state=None)
sample_mi = mi_pool.sample(n=counts["miscarriage"], replace=False, random_state=None)
sample_sh = sh_pool.sample(n=counts["harassment"], replace=False, random_state=None)

sample_all = pd.concat([sample_ab, sample_mi, sample_sh], ignore_index=True)
sample_all = sample_all.sample(frac=1.0).reset_index(drop=True)  # shuffle


In [7]:
# Save timestamped CSV
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
out_path = Path(f"unique_sample_{ts}.csv")
sample_all.to_csv(out_path, index=False)

# Update history
new_keys = sample_all["unique_key"].tolist()
history["used_keys"].extend(new_keys)
history["runs"].append({
    "timestamp": datetime.utcnow().isoformat() + "Z",
    "output_file": str(out_path),
    "counts": counts,
    "selected": len(new_keys)
})

with open(HISTORY_PATH, "w", encoding="utf-8") as f:
    json.dump(history, f, ensure_ascii=False, indent=2)

print(f"✅ Saved {len(sample_all)} posts to {out_path}")
print(f"Remaining after this run:")
print("  abortion:", len(ab_pool) - counts["abortion"])
print("  miscarriage:", len(mi_pool) - counts["miscarriage"])
print("  harassment:", len(sh_pool) - counts["harassment"])


✅ Saved 500 posts to unique_sample_20251020_214455.csv
Remaining after this run:
  abortion: 3236
  miscarriage: 552
  harassment: 3987


In [8]:
# Make a clean training label column (good for ML pipelines)
sample_all = sample_all.copy()
sample_all["label"] = sample_all["__dataset"]  # keep your original columns intact

# Create a training_sets folder
TRAIN_DIR = Path("training_sets")
TRAIN_DIR.mkdir(parents=True, exist_ok=True)

# Save a per-run training file (500 rows)
train_ts = datetime.now().strftime("%Y%m%d_%H%M%S")
train_csv = TRAIN_DIR / f"train_{train_ts}.csv"
sample_all.to_csv(train_csv, index=False)

# Also keep a stable "latest" pointer you can reference in code
latest_csv = TRAIN_DIR / "train_latest.csv"
sample_all.to_csv(latest_csv, index=False)

print(f"✅ Saved training set (500 rows): {train_csv}")
print(f"🔁 Also updated: {latest_csv}")

# (Optional) Save per-class training files for class-specific experiments
PER_CLASS_DIR = TRAIN_DIR / f"per_class_{train_ts}"
PER_CLASS_DIR.mkdir(parents=True, exist_ok=True)

for cls in sample_all["label"].unique():
    out_cls = PER_CLASS_DIR / f"{cls}_train_{train_ts}.csv"
    sample_all[sample_all["label"] == cls].to_csv(out_cls, index=False)
    print(f"• Saved {cls} subset to: {out_cls}")

# (Optional) Keep a cumulative union of everything ever sampled (good for audit/repro)
CUMULATIVE_CSV = TRAIN_DIR / "all_selected_so_far.csv"
if CUMULATIVE_CSV.exists():
    prev = pd.read_csv(CUMULATIVE_CSV, low_memory=False)
    # Use unique_key to de-dup
    combined = pd.concat([prev, sample_all], ignore_index=True)
    combined = combined.drop_duplicates(subset=["unique_key"])
else:
    combined = sample_all

combined.to_csv(CUMULATIVE_CSV, index=False)
print(f"📚 Cumulative selected-so-far updated: {CUMULATIVE_CSV}")


✅ Saved training set (500 rows): training_sets/train_20251020_214522.csv
🔁 Also updated: training_sets/train_latest.csv
• Saved miscarriage subset to: training_sets/per_class_20251020_214522/miscarriage_train_20251020_214522.csv
• Saved abortion subset to: training_sets/per_class_20251020_214522/abortion_train_20251020_214522.csv
• Saved harassment subset to: training_sets/per_class_20251020_214522/harassment_train_20251020_214522.csv
📚 Cumulative selected-so-far updated: training_sets/all_selected_so_far.csv


In [9]:
sample_all.head(10)

,id,subreddit,title,selftext,created_utc,url,Tags,__dataset,__source_file,unique_key,score,num_comments,link_flair_text,over_18,strategy,label
0,1o3a9pf,miscarriage,"My friend miscarried, what do I do?",My friend lives a few hours away and had a mis...,2025-10-10T19:19:59,https://www.reddit.com/r/Miscarriage/comments/...,NaN,miscarriage,miscarriage_related_posts.csv,1bbc312d8f9dca2d9d2d7ebefd4fb718be0465a7dd1e14...,6.0,20.0,support for someone who miscarried,False,new,miscarriage
1,1nrdmyk,miscarriage,Irregular bleeding after 3 weeks?,I had my first miscarriage on September 1st. ...,2025-09-26T21:32:51,https://www.reddit.com/r/Miscarriage/comments/...,NaN,miscarriage,miscarriage_related_posts.csv,9612314557bcd75705d954856eadb67a071fa93c3697df...,1.0,4.0,question/need help,False,new,miscarriage
2,1nutyvy,miscarriage,Heartbeat detected 4 days ago… gone?,3 days ago (sept 27) I had my first ultrasound...,2025-10-01T00:29:25,https://www.reddit.com/r/Miscarriage/comments/...,NaN,miscarriage,miscarriage_related_posts.csv,4ee1a3396763f7dd00b868fe2c161fb0fe39d8dc739871...,2.0,5.0,experience: first MC,False,new,miscarriage
3,cz6tt7,Parenting,"Ahhh, I love the smell of logic in the morning.",Super quick story. \n\nMy six year old tried t...,2019-09-03 16:03:06,https://www.reddit.com/r/Parenting/comments/cz...,NaN,abortion,abortion_data-updated - new_abortion_related_s...,69d8ec5cad574d1f08b74313a7eda5480dfd81c822ceff...,NaN,NaN,NaN,NaN,NaN,abortion
4,lu7mr4,assault,I was the perpetrator of child-on-child sexual...,"So, about 4 years ago, I was at my friend's ho...",2021-02-28 5:26:43,https://www.reddit.com/r/sexualassault/comment...,NaN,harassment,sexual-harrassment-data-updated - RelevantByTi...,e7cac983b6c0fc4c480265638d4e4598d6929bd95d87f9...,NaN,NaN,NaN,NaN,NaN,harassment
5,1nye7aq,miscarriage,When will be normal again,I just had a miscarriage 2 months ago (8 weeks...,2025-10-05T03:57:55,https://www.reddit.com/r/Miscarriage/comments/...,NaN,miscarriage,miscarriage_related_posts.csv,277fd65f5c3aaa898c952fab103e88c10269a60f726a5f...,3.0,3.0,question/need help,False,new,miscarriage
6,1o62lfp,miscarriage,Post d&c fever?,"Hi everyone, I had a d&c Tuesday last week, al...",2025-10-14T01:39:33,https://www.reddit.com/r/Miscarriage/comments/...,NaN,miscarriage,miscarriage_related_posts.csv,17015c268f0c54310bcb238d933d31c076375ff0c72f22...,3.0,3.0,experience: D&C,False,new,miscarriage
7,1nnn3y9,miscarriage,Retained blood clot,I experienced by first miscarriage at 8 weeks ...,2025-09-22T13:53:53,https://www.reddit.com/r/Miscarriage/comments/...,NaN,miscarriage,miscarriage_related_posts.csv,53fb225515298dd4631ed653ff84688b8777db5973e6cc...,1.0,3.0,experience: first MC,False,new,miscarriage
8,n6dkbl,assault,Would it be shitty of me to report an assault ...,So I(NB/F20) was groped by a friend(F21) at a ...,2021-05-06 18:08:40,https://www.reddit.com/r/sexualassault/comment...,NaN,harassment,sexual-harrassment-data-updated - RelevantByTi...,883ae6c8a6ea78c60d0e353d71b108cf268bac2ede205f...,NaN,NaN,NaN,NaN,NaN,harassment
9,1o5tsuz,miscarriage,After a blighted ovum and currently suffering ...,It wasnt enough in July for me to have an ovum...,2025-10-13T19:36:31,https://www.reddit.com/r/Miscarriage/comments/...,NaN,miscarriage,miscarriage_related_posts.csv,e804937c74633ddfe379ac3a3a0eeaada75a8fefc9c1bc...,25.0,24.0,vent,False,new,miscarriage
